In [5]:
import numpy as np

# Matriz de Confusão

### Verdadeiros Negativos

Val1

### Falsos Positivos

Identificação falsa de algo que era legítimo

### Falsos Negativos

Não identificação de problema

### Verdadeiros Positivos

## Acurácia

Quantidade de acertos em razão com o total de previsões;
Ela engana quando as classes são desbalanceadas (pende pra um lado) e quando um split é inviesado pra um lado (overfitting).

In [6]:
y_real = np.array([1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 1, 0])

acertos = (y_real == y_pred)

accuracy = acertos.mean()

## Precision

Quantidade de TP em relação ao total de TP + FP

Quando confiar no positivo

## Recall (Sensibilidade)

Quantidade de TP em relação ao total de TP + FN

## F1-Score

2 * (P * R) / (P + R), com P sendo Precision e R Recall.

Média harmônica é utilizada (expressa melhor a diferença entre os valores das duas outras métricas)

In [7]:
# True Positive

print(np.sum((y_pred == 1) & (y_real == 1)))

2


In [1]:
# Aula 6.5 · Exercício guiado: as métricas em dez etapas
# preveja: escreva o resultado esperado antes de executar
import numpy as np


# 01 · Carregar y_real e y_pred
# y_real = resposta certa, y_pred = o que o modelo previu
# classe 1 = spam, classe 0 = mensagem normal
y_real = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])
y_pred = np.array([1, 0, 1, 1, 1, 0, 0, 0, 1, 1])


# 02 · Acertos e accuracy · preveja: ____
# accuracy = acertos / total
acertos = (y_real == y_pred)
print("acertos:", acertos.sum())
print("accuracy:", acertos.mean())


# 03 · Contar TP, TN, FP e FN
# TP: era 1 e previu 1 | TN: era 0 e previu 0
# FP: era 0 e previu 1 | FN: era 1 e previu 0
def contar(y_real, y_pred):
    tp = np.sum((y_real == 1) & (y_pred == 1))
    tn = np.sum((y_real == 0) & (y_pred == 0))
    fp = np.sum((y_real == 0) & (y_pred == 1))
    fn = np.sum((y_real == 1) & (y_pred == 0))
    return tp, tn, fp, fn

tp, tn, fp, fn = contar(y_real, y_pred)
print("TP:", tp, "TN:", tn, "FP:", fp, "FN:", fn)


# 04 · Construir a matriz de confusão
# linhas = classe real, colunas = classe prevista
# [[TN, FP],
#  [FN, TP]]
def matriz_confusao(y_real, y_pred):
    tp, tn, fp, fn = contar(y_real, y_pred)
    return np.array([[tn, fp],
                     [fn, tp]])

print(matriz_confusao(y_real, y_pred))


# 05 · Implementar precision · preveja: ____
# precision = TP / (TP + FP)
def divide(num, den):
    return num / den if den != 0 else 0.0

def precision(tp, fp):
    return divide(tp, tp + fp)

p = precision(tp, fp)
print("precision:", p)


# 06 · Implementar recall
# recall = TP / (TP + FN)
def recall(tp, fn):
    return divide(tp, tp + fn)

r = recall(tp, fn)
print("recall:", r)


# 07 · Implementar F1-score · preveja: ____
# F1 = 2·P·R / (P + R)
def f1_score(p, r):
    return divide(2 * p * r, p + r)

print("F1:", f1_score(p, r))


# 08 · Estender para três classes
# 0 = Adelie, 1 = Chinstrap, 2 = Gentoo
# M[i, j] = reais da classe i previstos como j
y_real3 = np.array([0]*10 + [1]*5 + [2]*10)
y_pred3 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
                    0, 1, 1, 1, 2,
                    1, 2, 2, 2, 2, 2, 2, 2, 2, 2])

classes = np.unique(np.concatenate([y_real3, y_pred3]))
k = len(classes)

M = np.zeros((k, k), dtype=int)
for real, prev in zip(y_real3, y_pred3):
    M[real, prev] += 1
print(M)


# 09 · Montar o relatório de classificação
# one-vs-rest: TP = diagonal, FP = resto da coluna,
# FN = resto da linha, support = soma da linha
def razao(num, den):
    num, den = np.asarray(num, dtype=float), np.asarray(den, dtype=float)
    out = np.zeros_like(num)
    np.divide(num, den, out=out, where=den != 0)
    return out

tp3 = np.diag(M)
fp3 = M.sum(axis=0) - tp3      # coluna sem a diagonal
fn3 = M.sum(axis=1) - tp3      # linha sem a diagonal
prec3 = razao(tp3, tp3 + fp3)
rec3  = razao(tp3, tp3 + fn3)
f13   = razao(2 * prec3 * rec3, prec3 + rec3)
support = M.sum(axis=1)
acc3 = tp3.sum() / M.sum()

nomes = ["Adelie", "Chinstrap", "Gentoo"]
print(f"{'':>14}{'precision':>11}{'recall':>9}{'f1-score':>10}{'support':>9}")
for i in range(k):
    print(f"{nomes[i]:>14}{prec3[i]:>11.2f}{rec3[i]:>9.2f}{f13[i]:>10.2f}{support[i]:>9}")
print(f"{'accuracy':>14}{'':>20}{acc3:>10.2f}{support.sum():>9}")
print(f"{'macro avg':>14}{prec3.mean():>11.2f}{rec3.mean():>9.2f}{f13.mean():>10.2f}{support.sum():>9}")
print(f"{'weighted avg':>14}"
      f"{np.average(prec3, weights=support):>11.2f}"
      f"{np.average(rec3, weights=support):>9.2f}"
      f"{np.average(f13, weights=support):>10.2f}{support.sum():>9}")


# 10 · ROC e AUC do exemplo binário · preveja: ____
# TPR = TP / (TP + FN)    FPR = FP / (FP + TN)
# scores = probabilidades que o modelo deu para cada mensagem;
# o y_pred da etapa 01 é exatamente scores >= 0.5
scores = np.array([0.95, 0.10, 0.70, 0.55, 0.85, 0.38, 0.45, 0.25, 0.62, 0.78])

def curva_roc(y_real, scores):
    ordem = np.argsort(-scores, kind="mergesort")
    y, s = y_real[ordem], scores[ordem]
    fim = np.r_[np.where(np.diff(s) != 0)[0], len(s) - 1]
    tp = np.cumsum(y)[fim]
    fp = (fim + 1) - tp
    P, N = y.sum(), len(y) - y.sum()
    tpr = np.r_[0, tp] / P
    fpr = np.r_[0, fp] / N
    return fpr, tpr

# AUC pela regra dos trapézios: A = (x2 - x1) * (y1 + y2) / 2
def auc(fpr, tpr):
    larg = fpr[1:] - fpr[:-1]
    alt = (tpr[1:] + tpr[:-1]) / 2
    return np.sum(larg * alt)

fpr, tpr = curva_roc(y_real, scores)
print("FPR:", fpr)
print("TPR:", tpr)
print("AUC:", auc(fpr, tpr))

acertos: 7
accuracy: 0.7
TP: 4 TN: 3 FP: 2 FN: 1
[[3 2]
 [1 4]]
precision: 0.6666666666666666
recall: 0.8
F1: 0.7272727272727272
[[8 2 0]
 [1 3 1]
 [0 1 9]]
                precision   recall  f1-score  support
        Adelie       0.89     0.80      0.84       10
     Chinstrap       0.50     0.60      0.55        5
        Gentoo       0.90     0.90      0.90       10
      accuracy                          0.80       25
     macro avg       0.76     0.77      0.76       25
  weighted avg       0.82     0.80      0.81       25
FPR: [0.  0.  0.  0.2 0.2 0.2 0.4 0.4 0.6 0.8 1. ]
TPR: [0.  0.2 0.4 0.4 0.6 0.8 0.8 1.  1.  1.  1. ]
AUC: 0.8400000000000001
